# Imports

In [ ]:
import pandas as pd

# Load census data

## England & Wales

In [ ]:
ew = pd.read_csv('../census_data/england_wales_2021.csv')

In [ ]:
ew.columns = ['constituency_code', 'constituency_name', 'religion_code', 'religion', 'count']

## Northern Ireland

In [ ]:
ni = pd.read_csv('../census_data/northern_ireland_2021.csv')

In [ ]:
ni.columns = ['constituency_code', 'constituency_name', 'religion_code', 'religion', 'count']

## Scotland

In [ ]:
sc = pd.read_csv('../census_data/scotland_2022.csv', header=7).iloc[1:58, [0,2,3,4,5,6,7,8,9,10,11,12,13]]

In [ ]:
sc.rename(columns={'Religion - 12 groups': 'constituency_name'}, inplace=True)

In [ ]:
sc = sc.melt(id_vars='constituency_name', var_name='religion', value_name='count')

# Load constituency to LAD map

In [ ]:
constituency_LAD_map = pd.read_csv('../geo_data/constituency_LAD_map.csv')

# Aggregate granular Christian categories

In [ ]:
christian_cats = {
    'Catholic',
    'Christian',
    'Church of Ireland',
    'Church of Scotland',
    'Methodist Church in Ireland',
    'Other Christian',
    'Other Christian (including Christian related)',
    'Presbyterian Church in Ireland',
    'Roman Catholic',
    'Methodist',
    'Presbyterian',
}

In [ ]:
dfs = []
for df in [ew, ni, sc]:
    df['religion2'] = df['religion'].apply(lambda x: 'Christian' if x in christian_cats else x)
    dfs.append(df.groupby(['constituency_name', 'religion2'])['count'].sum())

# Load predictions data

In [ ]:
ew_preds = pd.read_csv('../LAD_projections/ew_preds.csv').drop(columns=['sex', 'age_band']).groupby(['year', 'geo_code']).sum()
sc_preds = pd.read_csv('../LAD_projections/sc_preds.csv').drop(columns=['sex', 'age_band']).groupby(['year', 'geo_code']).sum()
ni_preds = pd.read_csv('../LAD_projections/ni_preds.csv').drop(columns=['sex', 'age_band']).groupby(['year', 'geo_code']).sum()

In [ ]:
sc_preds = sc_preds.rename(columns=lambda x: 'Christian' if x in christian_cats else x)
sc_preds = sc_preds.T.groupby(sc_preds.columns).sum().T

In [ ]:
ni_preds = ni_preds.rename(columns=lambda x: 'Christian' if x in christian_cats else x)
ni_preds = ni_preds.T.groupby(ni_preds.columns).sum().T

## Actuals at constituency level 2021

In [ ]:
sc_actuals_2021 = sc.replace({'Pagan': 'Other religion'}).groupby(['constituency_name', 'religion2'])['count'].sum().unstack()

In [ ]:
ni_actuals_2021 = ni.groupby(['constituency_name', 'religion2'])['count'].sum().unstack().rename(columns={'Religion not stated': 'Not stated'})

In [ ]:
ew_actuals_2021 = ew.groupby(['constituency_name', 'religion2'])['count'].sum().unstack().drop(columns='Does not apply')

## Make estimated projections at constituency level from weighted sum of LADs 2021

In [ ]:
def estimate(LAD_map, LAD_preds, year, rel_cols):
    df = LAD_map.join(LAD_preds.loc[year], on='geo_code', how='inner').set_index('constituency_name')
    return df[rel_cols].mul(df['overlap_geo_frac_norm'], axis=0).groupby('constituency_name').sum() # changed from frac_overlap_norm

### Scotland

In [ ]:
sc_cols = ['Buddhist','Christian','Hindu','Jewish','Muslim','No religion','Other religion','Religion not stated','Sikh']
sc_est_2021 = estimate(constituency_LAD_map, sc_preds, 2021, sc_cols)
sc_actuals_2021_pct = sc_actuals_2021.apply(lambda x: x/x.sum(), axis=1)
sc_est_2021_pct = sc_est_2021.apply(lambda x: x/x.sum(), axis=1)

#### Calculate ratio of estimate to actual (known) census data for constituency

In [ ]:
sc_ratios = (sc_actuals_2021_pct / sc_est_2021_pct).dropna()

In [ ]:
# rough population check
sc_est_2021.sum(axis=1).sum()

### NI

In [ ]:
ni_cols = ['Christian','No religion','Other religions','Not stated']
ni_est_2021 = estimate(constituency_LAD_map, ni_preds, 2021, ni_cols)
ni_actuals_2021_pct = ni_actuals_2021.apply(lambda x: x/x.sum(), axis=1)
ni_est_2021_pct = ni_est_2021.apply(lambda x: x/x.sum(), axis=1)

#### Calculate ratio of estimate to actual (known) census data for constituency

In [ ]:
ni_ratios = (ni_actuals_2021_pct / ni_est_2021_pct)

In [ ]:
# rough population check
ni_est_2021.sum().sum()

### England & Wales

In [ ]:
ew_cols = ['Buddhist','Christian','Hindu','Jewish','Muslim','No religion','Not answered','Other religion','Sikh']
ew_est_2021 = estimate(constituency_LAD_map, ew_preds, 2021, ew_cols)
ew_actuals_2021_pct = ew_actuals_2021.apply(lambda x: x/x.sum(), axis=1)
ew_est_2021_pct = ew_est_2021.apply(lambda x: x/x.sum(), axis=1)

#### Calculate ratio of estimate to actual (known) census data for constituency

In [ ]:
ew_ratios = (ew_actuals_2021_pct / ew_est_2021_pct).dropna()

In [ ]:
# rough population check
ew_est_2021.sum().sum()

# Adjust projections using ratios of estimated to known data for 2021

In [ ]:
def adjusted_estimate(LAD_map, LAD_preds, year, rel_cols, ratio):
    df = (estimate(LAD_map, LAD_preds, year, rel_cols) * ratio).dropna()
    df['biggest'] = df.idxmax(axis=1).replace({'No religion': 'Non-religious'})
    df['share'] = df[rel_cols].max(axis=1) / df[rel_cols].sum(axis=1)
    df['majority'] = df['share'] > 0.5
    df['category'] = df.apply(lambda row: row['biggest'] + (' majority' if row['majority'] else ' plurality'), axis=1)
    return df

## Write projections for animated GIF to disk

In [ ]:
df_preds = {}
for yr in range(2021, 2032):
    df_preds[yr] = pd.concat([
        adjusted_estimate(constituency_LAD_map, ew_preds, yr, ew_cols, ew_ratios).iloc[:, -4:],
        adjusted_estimate(constituency_LAD_map, sc_preds, yr, sc_cols, sc_ratios).iloc[:, -4:],
        adjusted_estimate(constituency_LAD_map, ni_preds, yr, ni_cols, ni_ratios).iloc[:, -4:],
    ])
    df_preds[yr].to_csv(f'../article_charts/population_animation/projections_{yr}.csv')

# Isolate electorate by removing under 18s

In [ ]:
drop_bands = [
    '0-0','0-1','0-2','0-3','0-4',
    '1-5','2-6','3-7','4-8','5-9',
    '6-10','7-11','8-12','9-13','10-14',
    '11-15','12-16','13-17',
]

def just_voters(preds_csv):
    df_ret = (
        pd.read_csv(preds_csv)
        .drop(columns=['sex'])
        .groupby(['year', 'geo_code', 'age_band']).sum()
        .reset_index(['year','geo_code'])
        .drop(index=drop_bands)
        .reset_index()
        .set_index(['age_band', 'year', 'geo_code'])
    )
    df_ret.loc[['14-18']] = df_ret.loc[['15-19']] * 1/5
    df_ret.loc[['15-19']] = df_ret.loc[['15-19']] * 2/5
    df_ret.loc[['16-20']] = df_ret.loc[['15-19']] * 3/5
    df_ret.loc[['17-21']] = df_ret.loc[['15-19']] * 4/5
    return (
        df_ret
        .rename(index={
            '14-18':'18-18',
            '15-19':'18-19',
            '16-20':'18-20',
            '17-21':'18-21',
        })
        .reset_index(['year', 'geo_code'])
        .groupby(['year', 'geo_code']).sum()
    )

## Make projections for voters

In [ ]:
ew_preds_voters = just_voters('../LAD_projections/ew_preds.csv')
sc_preds_voters = just_voters('../LAD_projections/sc_preds.csv')
ni_preds_voters = just_voters('../LAD_projections/ni_preds.csv')

In [ ]:
sc_preds_voters = sc_preds_voters.rename(columns=lambda x: 'Christian' if x in christian_cats else x)
sc_preds_voters = sc_preds_voters.T.groupby(sc_preds_voters.columns).sum().T

In [ ]:
ni_preds_voters = ni_preds_voters.rename(columns=lambda x: 'Christian' if x in christian_cats else x)
ni_preds_voters = ni_preds_voters.T.groupby(ni_preds_voters.columns).sum().T

In [ ]:
df_preds_voters = {}
for yr in range(2021, 2032):
    df_preds_voters[yr] = pd.concat([
        adjusted_estimate(constituency_LAD_map, ew_preds_voters, yr, ew_cols, ew_ratios).iloc[:, -4:],
        adjusted_estimate(constituency_LAD_map, sc_preds_voters, yr, sc_cols, sc_ratios).iloc[:, -4:],
        adjusted_estimate(constituency_LAD_map, ni_preds_voters, yr, ni_cols, ni_ratios).iloc[:, -4:],
    ])

# Load current party

In [ ]:
mps = pd.read_csv('mps.csv')

In [ ]:
mps['mp_name'] = mps['First name'].str.cat(mps['Last name'], sep=' ')

In [ ]:
current_parties = df_preds_voters[2029].join(mps[['Constituency', 'Party', 'mp_name']].set_index('Constituency'))[['Party', 'mp_name']].fillna('Vacant')

In [ ]:
# this constituency is spelled with a Welsh ŵ character in the original dataset
current_parties.loc['Montgomeryshire and Glyndwr', 'Party'] = 'Labour'

In [ ]:
current_parties = current_parties.replace({'Labour/Co-operative': 'Labour'})

# Charts

## Stacked column chart

In [ ]:
voter_pop_cats = pd.concat([df_preds_voters[k]['category'].value_counts().rename(k) for k in df_preds_voters], axis='columns')

In [ ]:
voter_pop_cats

In [ ]:
voter_pop_cats.T.to_csv('../article_charts/stacked_column.csv')

## Interactive hex map and party summary table

In [ ]:
df_preds_voters_2024 = pd.concat([
    adjusted_estimate(constituency_LAD_map, ew_preds_voters, 2024, ew_cols, ew_ratios),
    adjusted_estimate(constituency_LAD_map, sc_preds_voters, 2024, sc_cols, sc_ratios),
    adjusted_estimate(constituency_LAD_map, ni_preds_voters, 2024, ni_cols, ni_ratios),
])

In [ ]:
df_preds_voters_2029 = pd.concat([
    adjusted_estimate(constituency_LAD_map, ew_preds_voters, 2029, ew_cols, ew_ratios),
    adjusted_estimate(constituency_LAD_map, sc_preds_voters, 2029, sc_cols, sc_ratios),
    adjusted_estimate(constituency_LAD_map, ni_preds_voters, 2029, ni_cols, ni_ratios),
])

### Aggregate other religion and not answered categories

In [ ]:
other_cols = [
    'Buddhist',
    'Hindu',
    'Jewish',
    'Muslim',
    'Other religion',
    'Sikh',
    'Other religions',
]

In [ ]:
na_cols = [
    'Not answered',
    'Religion not stated',
    'Not stated'
]

In [ ]:
df_preds_voters_2024['Other'] = df_preds_voters_2024[other_cols].sum(axis='columns')
df_preds_voters_2029['Other'] = df_preds_voters_2029[other_cols].sum(axis='columns')

df_preds_voters_2024['NA'] = df_preds_voters_2024[na_cols].sum(axis='columns')
df_preds_voters_2029['NA'] = df_preds_voters_2029[na_cols].sum(axis='columns')

In [ ]:
keep_cols = [
    'Christian',
    'No religion',
    'category',
    'Other',
    'NA',
]

In [ ]:
df_preds_voters_2024 = df_preds_voters_2024[keep_cols].rename(columns={'category': 'category_2024'})
df_preds_voters_2029 = df_preds_voters_2029[keep_cols].rename(columns={'category': 'category_2029'})

### Convert to percentages

In [ ]:
rel_cols = ['Christian', 'No religion', 'Other', 'NA']

In [ ]:
df_preds_voters_2024.loc[:, rel_cols] = df_preds_voters_2024.loc[:, rel_cols].div(df_preds_voters_2024.loc[:, rel_cols].sum(axis='columns'), axis='rows') * 100
df_preds_voters_2029.loc[:, rel_cols] = df_preds_voters_2029.loc[:, rel_cols].div(df_preds_voters_2029.loc[:, rel_cols].sum(axis='columns'), axis='rows') * 100

### Join on current party data

In [ ]:
df_preds_voters_2029 = df_preds_voters_2029.join(current_parties)

### Rename columns

In [ ]:
df_preds_voters_2024.rename(columns={'Christian': 'christian_2024', 'No religion': 'no_religion_2024', 'Other': 'other_2024', 'NA': 'na_2024'}, inplace=True)
df_preds_voters_2029.rename(columns={'Christian': 'christian_2029', 'No religion': 'no_religion_2029', 'Other': 'other_2029', 'NA': 'na_2029'}, inplace=True)

### Join 2024 and 2029 data

In [ ]:
df_preds_voters_2029 = df_preds_voters_2029.join(df_preds_voters_2024)

In [ ]:
df_preds_voters_2029.loc['North Down', 'Party'] = 'NI Independent'

## Parties summary table

In [ ]:
cat_map = {
    'Non-religious majority': 'None',
    'Non-religious plurality': 'None',
    'Christian plurality': 'Christian',
    'Muslim plurality': 'Other',
    'Muslim majority': 'Other',
    'Christian majority': 'Christian',
    'Hindu plurality': 'Other',
}

In [ ]:
party_map = {
    'Independent': 'Other',
    'Speaker': 'Other',
    'Your Party': 'Other',
    'Restore Britain': 'Other',
    'Vacant': 'Other',
    'Independent': 'Other',
    'DUP': 'Northern Ireland',
    'Sinn Féin': 'Northern Ireland',
    'Social Democratic and Labour Party': 'Northern Ireland',
    'Alliance': 'Northern Ireland',
    'Traditional Unionist Voice': 'Northern Ireland',
    'UUP': 'Northern Ireland',
    'NI Independent': 'Northern Ireland',
}

In [ ]:
cat_2029 = df_preds_voters_2029.replace(cat_map).replace(party_map).groupby(['Party', 'category_2029']).size().unstack().fillna(0)

In [ ]:
cat_2024 = df_preds_voters_2029.replace(cat_map).replace(party_map).groupby(['Party', 'category_2024']).size().unstack().fillna(0)

In [ ]:
for_table = pd.concat([cat_2024, cat_2029], axis=1, keys=['2024 Plurality','2029 Plurality']).astype(int)

In [ ]:
for_table.to_csv('../article_charts/parties_table.csv')

## Interactive hex map data

In [ ]:
party_map2 = {
    'Liberal Democrat': 'Lib Dem',
    'Scottish National Party': 'SNP',
    'Social Democratic and Labour Party': 'SDLP',
    'Traditional Unionist Voice': 'TUV',
}

In [ ]:
df_preds_voters_2029.replace(party_map2).to_csv('../article_charts/interactive_hex_map.csv')

# Detailed table

In [ ]:
mp_table = pd.concat([df_preds_voters[k][['share', 'category']].rename(columns={'share': k, 'category': f'category_{k}'}) for k in [2024,2029]], axis='columns')
mp_table[['mp_name', 'party']] = df_preds_voters_2029.replace(party_map2)[['mp_name', 'Party']]
mp_table.loc[:, [2024,2029]] *= 100
mp_table = mp_table.iloc[:, [-1, -2, 1, 0, 3, 2]]

In [ ]:
party_map3 = {
    'Labour': 'LAB',
    'Conservative': 'CON',
    'Reform UK': 'RFM',
    'Lib Dem': 'LDM',
    'Independent': 'IND',
    'Green': 'GRN',
    'Plaid Cymru': 'PC',
    'Speaker': 'SPK',
    'Your Party': 'YP',
    'Restore Britain': 'REST',
    'Vacant': 'VAC',
    'SNP': 'SNP',
    'DUP': 'DUP',
    'Sinn Féin': 'SF',
    'SDLP': 'SDLP',
    'Alliance': 'APNI',
    'TUV': 'TUV',
    'NI Independent': 'IND',
    'UUP': 'UUP'
}

In [ ]:
mp_table['party'] = mp_table['party'].map(party_map3)

In [ ]:
mp_table.to_csv('../article_charts/detailed_mp_table.csv')